# BERTopic - topic modelling of news descriptions 

[BERTopic](https://maartengr.github.io/BERTopic/index.html) is a library for doing topic modelling with BERT language model. Maybe you have heard of an alternative - LDA topic modelling - this approach is based on different technical point of view than LDA.  BERTopic is based on dimensionality reduction and clustering of BERT sentence embeddings. More details of the algorithm you can find [here](https://maartengr.github.io/BERTopic/algorithm/algorithm.html#4-bag-of-words). 

First, we have to import some useful libraries in addition to BERTopic. We import pandas and numpy for processing the data. If you haven't installed these libraries already, you can check [install pandas](https://pandas.pydata.org/docs/getting_started/install.html) and [install numpy](https://numpy.org/install/) descriptions. 

In [1]:
import pandas as pd 
import numpy as np
from bertopic import BERTopic

/Users/huhtis/Teaching/datatie2025/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/huhtis/Teaching/datatie2025/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Our dataset is a news headline & abstract dataset from  [Kaggle](https://www.kaggle.com/datasets/rmisra/news-category-dataset). The dataset includes around 210 000 records, each record including multiple attributes such as authors, headline, link to the original article, and description of the news article. In our project, we are mainly interested in the descriptions. 

Next, we read our dataset with the suitable command. When we print the length of the dataframe, we see that it is quite big, almost 210000 items, as described above. We want to reduce the size of the dataset and we do that with indexes. 

In [2]:
df_full = pd.read_json("News_Category_Dataset_v3.json", lines=True)
print(len(df_full))
df = df_full[0:1000]

209527


In [3]:
df_full.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [4]:

df_categories = pd.DataFrame(df_full.category.value_counts())
df_categories.head()
df_categories.to_csv('data/categories.csv')

## Training the model 

Finally we get into the training of the model. We define our BERTopic model as `model`, take a look to short descriptions by printing the first of them and then define a list of documents from the short descriptions. We start training the model with `fit_transform` function. 

In [5]:
print(df['short_description'][0])
documents = df['short_description'].to_list()

model = BERTopic(verbose=True)

topics, probabilities = model.fit_transform(documents)

2025-04-04 12:48:57,415 - BERTopic - Embedding - Transforming documents to embeddings.


Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.


Batches: 100%|██████████| 32/32 [00:01<00:00, 19.65it/s]
2025-04-04 12:49:02,480 - BERTopic - Embedding - Completed ✓
2025-04-04 12:49:02,480 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-04-04 12:49:05,896 - BERTopic - Dimensionality - Completed ✓
2025-04-04 12:49:05,897 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-04 12:49:05,912 - BERTopic - Cluster - Completed ✓
2025-04-04 12:49:05,914 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-04 12:49:05,933 - BERTopic - Representation - Completed ✓


Let's take a look to the topics. We see, that the first topic, is not so understandable. It is mostly about pronouns and conjunctions. The one weird word among others is "dog". Another topic is clearly about war in Ukraine, we can see it from words such as "ukraine", "war" and "kyiv". 

In [6]:
print(model.get_topic(0))
print("-----------------")
print(model.get_topic(1))

[('the', np.float64(0.05367392594507008)), ('in', np.float64(0.03721129711600945)), ('to', np.float64(0.03449848804508429)), ('of', np.float64(0.03343395573736291)), ('and', np.float64(0.027688157143058886)), ('was', np.float64(0.02529800286170105)), ('former', np.float64(0.024962075877659345)), ('trump', np.float64(0.023299533676578502)), ('said', np.float64(0.02194837577000164)), ('is', np.float64(0.021947476848039826))]
-----------------
[('the', np.float64(0.06542276910402886)), ('her', np.float64(0.04141459261308294)), ('of', np.float64(0.03543384768600848)), ('and', np.float64(0.03379546767020974)), ('to', np.float64(0.030989858265808235)), ('actor', np.float64(0.029972049576001917)), ('star', np.float64(0.029094093834421988)), ('music', np.float64(0.02836057484942657)), ('in', np.float64(0.02720783128718139)), ('that', np.float64(0.026390206897776516))]


## Topic visualization

Next we visualize topics based on their size (number of documents belonging to a topic) and how near topics are each others based on the words of the topics. With sliding the bar, we can focus on specific topics. 

In [7]:
model.visualize_topics()

## Topic similarity heatmap

We can also visualize how similar topics are to each others. This kind of heatmap is in general a common way to visualize data in Data Science applications. In the diagonal with blue color, we can see, that topics are fully similar (similarity score 1) with each others. Then we read the similarities by taking one item from x-axis and one item from y-axis. The similarity is calculated as cosine similarity of topic embeddings. 

In [8]:
model.visualize_heatmap()

## Topics for new documents 

Now let's take some documents that the topic model hasn't seen yet. Let's take 100 more documents from the full dataset and get the probabilities of different topics for them with `fit_transform` function.  

In [9]:
docs_test = df_full[1000:1050]['short_description'].to_list()

topics, probs = model.fit_transform(docs_test)
df_topics = pd.DataFrame({'topic': topics, 'document': docs_test})

df_topics.head()

2025-04-04 12:49:10,021 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 2/2 [00:00<00:00,  7.02it/s]
2025-04-04 12:49:10,311 - BERTopic - Embedding - Completed ✓
2025-04-04 12:49:10,311 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-04 12:49:10,348 - BERTopic - Dimensionality - Completed ✓
2025-04-04 12:49:10,349 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-04 12:49:10,350 - BERTopic - Cluster - Completed ✓
2025-04-04 12:49:10,351 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-04 12:49:10,355 - BERTopic - Representation - Completed ✓


,topic,document
0,-1,Jeff Zients and his deputy Natalie Quillian wi...
1,-1,The European Space Agency confirmed Thursday t...
2,-1,"Jobless claims fell by 15,000 to 214,000 for t..."
3,-1,Taoiseach Micheál Martin was at the Ireland Fu...
4,-1,The law also sought to explicitly ban same-sex...


In [14]:
df_topics.topic.value_counts()

topic
-1    50
Name: count, dtype: int64

We can see that topics are represented as numbers. Now let's remind ourselves of how to take a look to topics and print a document and it's topic words. 

In [10]:
for index, row in df_topics.iterrows():
    print(row["document"])
    print(model.get_topic(row["topic"]))

Jeff Zients and his deputy Natalie Quillian will leave the administration next month. Jha, the dean of the Brown University School of Public Health, will take over.
[('the', np.float64(0.1892691513786912)), ('to', np.float64(0.11744834957423995)), ('of', np.float64(0.10253350481488481)), ('and', np.float64(0.09731565291761048)), ('in', np.float64(0.07787739774068406)), ('russias', np.float64(0.04204405699984884)), ('with', np.float64(0.04204405699984884)), ('is', np.float64(0.04204405699984884)), ('invasion', np.float64(0.03829493604331553)), ('have', np.float64(0.03829493604331553))]
The European Space Agency confirmed Thursday that it’s indefinitely suspending its ExoMars rover mission with partner Roscosmos, Russia’s state space corporation.
[('the', np.float64(0.1892691513786912)), ('to', np.float64(0.11744834957423995)), ('of', np.float64(0.10253350481488481)), ('and', np.float64(0.09731565291761048)), ('in', np.float64(0.07787739774068406)), ('russias', np.float64(0.0420440569998

Let's make our printing a bit nicer by dropping the probabilities of topics from printing. If `zip` function isn't familiar for you, you can read from it [here](https://www.programiz.com/python-programming/methods/built-in/zip), from the example 3. So, in our case, `zip` function unpacks the tuples into two lists, where the first list includes topic words. 

Now we have a nicer printing for documents and their topic words. 

In [11]:
for index, row in df_topics.iterrows():
    print(row["document"])
    print("Topic words ", list(zip(*model.get_topic(row["topic"])))[0])
    print("---------------------------------------")

Jeff Zients and his deputy Natalie Quillian will leave the administration next month. Jha, the dean of the Brown University School of Public Health, will take over.
Topic words  ('the', 'to', 'of', 'and', 'in', 'russias', 'with', 'is', 'invasion', 'have')
---------------------------------------
The European Space Agency confirmed Thursday that it’s indefinitely suspending its ExoMars rover mission with partner Roscosmos, Russia’s state space corporation.
Topic words  ('the', 'to', 'of', 'and', 'in', 'russias', 'with', 'is', 'invasion', 'have')
---------------------------------------
Jobless claims fell by 15,000 to 214,000 for the week ending March 12, down from the previous week's 229,000.
Topic words  ('the', 'to', 'of', 'and', 'in', 'russias', 'with', 'is', 'invasion', 'have')
---------------------------------------
Taoiseach Micheál Martin was at the Ireland Funds 30th National Gala at the National Building Museum in Washington when he learned of his diagnosis.
Topic words  ('the',

## Specify number of topics 

Number of wanted topics can be also specified when defining the model. The algorithm then combines similar topics with each others. Let's specify the number of wanted topics and then run fitting the model to our documents again. We can see that the topics are quite similar than when running the model without specifying the number of topics. It is though an unsolved problem, how many topics should be used in topic modelling. 

In [12]:
model = BERTopic(nr_topics=10, verbose=True, calculate_probabilities=True) 
topics, probabilities = model.fit_transform(documents)

model.visualize_topics()

2025-04-04 12:49:10,376 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 32/32 [00:00<00:00, 61.81it/s]
2025-04-04 12:49:12,655 - BERTopic - Embedding - Completed ✓
2025-04-04 12:49:12,655 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-04 12:49:13,398 - BERTopic - Dimensionality - Completed ✓
2025-04-04 12:49:13,398 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-04 12:49:13,424 - BERTopic - Cluster - Completed ✓
2025-04-04 12:49:13,425 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-04-04 12:49:13,443 - BERTopic - Representation - Completed ✓
2025-04-04 12:49:13,444 - BERTopic - Topic reduction - Reducing number of topics
2025-04-04 12:49:13,447 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-04 12:49:13,464 - BERTopic - Representation - Completed ✓
2025-04-04 12:49:13,465 - BERTopic - Topic reduction - Redu

## Visualize topic probabilities for individual document 

Sometimes we are interested in digging deeper into individual documents. We can do it with `visualize_distribution` function. The function visualizes topic distribution for one document. Let's do it for the first document. 

In [13]:
print(documents[0])
model.visualize_distribution(probabilities[0])

Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.


TODO: Let's pick a particular news category, for example technology, and train a model usign the dataset. Explore the topics for the particular category and come up with an interesting insight that would benefit our organization. Students learning data science methods in the AI age, for example.